In [13]:
"""
Gene-to-m/z Rotation Analysis - Separate Scores Only
====================================================

APPROACH:
1. Loop angles 0-360.
2. For each gene, iterate ALL m/z features.
3. For each score type (Value Correlation, Radial, etc.), find the m/z that maximizes THAT specific score.
4. Store the 'Best Match' for each score type side-by-side in one CSV row.
"""
import numpy as np
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from scipy.interpolate import griddata
import pandas as pd
import os
import warnings
from dataclasses import dataclass

warnings.filterwarnings('ignore')

# =============================================================================
# DATA CONFIGURATION - VERIFY THESE PATHS
# =============================================================================

RNA_PIXEL_SIZE = 55
MSI_PIXEL_SIZE = 60

# NOTE: Double check these folders exist on your system!
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/dummy_data_8/msi/"
RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/dummy_data_8/rna/"

MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = [
    "Gene_Gradient_X", 
    "Gene_Gradient_Y",       # Vertical gradient
    "Gene_Radial_Center", 
    "Gene_Blob_TopRight", 
    "Gene_Stripes_Vertical",
    "Gene_Checkerboard",     # Grid pattern
    "Gene_Ring_Donut",       # A distinct ring around the center
    "Gene_Spots_Grid"        # Polka dot pattern
]  


In [14]:

# =============================================================================
# UTILS & DESCRIPTORS
# =============================================================================

def rotate_coords(coords: np.ndarray, angle_degrees: float) -> np.ndarray:
    theta = np.radians(angle_degrees)
    c, s = np.cos(theta), np.sin(theta)
    rotation_matrix = np.array([[c, -s], [s, c]])
    center = coords.mean(axis=0)
    centered = coords - center
    return (centered @ rotation_matrix.T) + center

def align_bounds(source_coords: np.ndarray, target_coords: np.ndarray) -> np.ndarray:
    s_min, s_max = source_coords.min(axis=0), source_coords.max(axis=0)
    t_min, t_max = target_coords.min(axis=0), target_coords.max(axis=0)
    scale = (t_max - t_min) / (s_max - s_min + 1e-8)
    return (source_coords - s_min) * scale + t_min

def compute_value_histogram(values, n_bins=50):
    hist, _ = np.histogram(values, bins=n_bins, density=True)
    return hist / (hist.sum() + 1e-8)

def compute_spatial_histogram(coords, values, n_bins=10):
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    hist = np.zeros((n_bins, n_bins))
    counts = np.zeros((n_bins, n_bins))
    for i in range(len(coords)):
        x = min(int(norm[i, 0] * n_bins), n_bins - 1)
        y = min(int(norm[i, 1] * n_bins), n_bins - 1)
        hist[y, x] += values[i]
        counts[y, x] += 1
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    if hist.max() > hist.min(): hist = (hist - hist.min()) / (hist.max() - hist.min())
    return hist

def compute_radial_profile(coords, values, n_rings=10):
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    centroid = norm.mean(axis=0)
    dists = np.linalg.norm(norm - centroid, axis=1)
    max_d = dists.max()
    prof = np.zeros(n_rings)
    cnts = np.zeros(n_rings)
    for i in range(len(coords)):
        r = min(int(dists[i] / max_d * n_rings), n_rings - 1)
        prof[r] += values[i]
        cnts[r] += 1
    prof = np.divide(prof, cnts, where=cnts > 0, out=np.zeros_like(prof))
    if prof.max() > prof.min(): prof = (prof - prof.min()) / (prof.max() - prof.min())
    return prof

def compute_quadrant_stats(coords, values, n_div=3):
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    stats = []
    for qx in range(n_div):
        for qy in range(n_div):
            xm, xM = qx/n_div, (qx+1)/n_div
            ym, yM = qy/n_div, (qy+1)/n_div
            mask = (norm[:,0]>=xm)&(norm[:,0]<xM)&(norm[:,1]>=ym)&(norm[:,1]<yM)
            if mask.sum()>0: stats.extend([np.mean(values[mask]), np.std(values[mask])])
            else: stats.extend([0,0])
    stats = np.array(stats)
    if stats.max() > stats.min(): stats = (stats - stats.min()) / (stats.max() - stats.min())
    return stats

def compute_morans_i(coords, values, k=8):
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    nn = NearestNeighbors(n_neighbors=k+1).fit(norm)
    _, idx = nn.kneighbors(norm)
    mean_v = values.mean()
    denom = np.sum((values - mean_v)**2)
    if denom==0: return 0
    numer = 0
    w = 0
    for i in range(len(values)):
        for j in idx[i, 1:]:
            numer += (values[i]-mean_v)*(values[j]-mean_v)
            w+=1
    return (len(values)/w)*(numer/denom) if w>0 else 0

@dataclass
class SpatialSignature:
    sample_id: str
    feature_name: str
    feature_type: str
    node_importance: np.ndarray
    value_histogram: np.ndarray = None
    spatial_histogram: np.ndarray = None
    radial_profile: np.ndarray = None
    quadrant_stats: np.ndarray = None
    morans_i: float = 0.0
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None
    aligned_coordinates: np.ndarray = None

def compute_all_scores(gene_sig, mz_sig, grid_res=50):
    scores = {}
    g_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None else gene_sig.coordinates
    m_coords = mz_sig.coordinates
    
    x_min = min(g_coords[:,0].min(), m_coords[:,0].min())
    x_max = max(g_coords[:,0].max(), m_coords[:,0].max())
    y_min = min(g_coords[:,1].min(), m_coords[:,1].min())
    y_max = max(g_coords[:,1].max(), m_coords[:,1].max())
    
    gx, gy = np.meshgrid(np.linspace(x_min, x_max, grid_res), np.linspace(y_min, y_max, grid_res))
    
    g_grid = griddata(g_coords, gene_sig.raw_values, (gx, gy), method='linear')
    m_grid = griddata(m_coords, mz_sig.raw_values, (gx, gy), method='linear')
    
    mask = ~(np.isnan(g_grid) | np.isnan(m_grid))
    if mask.sum() > 10:
        val_c, _ = pearsonr(g_grid[mask], m_grid[mask])
        scores['Value_Correlation'] = val_c if not np.isnan(val_c) else 0
    else:
        scores['Value_Correlation'] = 0
        
    g_imp_grid = griddata(g_coords, gene_sig.node_importance, (gx, gy), method='linear')
    m_imp_grid = griddata(m_coords, mz_sig.node_importance, (gx, gy), method='linear')
    mask_imp = ~(np.isnan(g_imp_grid) | np.isnan(m_imp_grid))
    
    if mask_imp.sum() > 0:
        gi = g_imp_grid.copy()
        mi = m_imp_grid.copy()
        gi[np.isnan(gi)] = 0
        mi[np.isnan(mi)] = 0
        gi /= (gi.max()+1e-8)
        mi /= (mi.max()+1e-8)
        scores['Importance_IoU'] = np.minimum(gi, mi).sum() / (np.maximum(gi, mi).sum() + 1e-8)
        imp_c, _ = pearsonr(gi[mask_imp], mi[mask_imp])
        scores['Importance_Correlation'] = imp_c if not np.isnan(imp_c) else 0
    else:
        scores['Importance_IoU'] = 0
        scores['Importance_Correlation'] = 0

    def safe_corr(a, b):
        if a.std() > 0 and b.std() > 0:
            c, _ = pearsonr(a.flatten(), b.flatten())
            return c if not np.isnan(c) else 0
        return 0

    scores['Value_Hist_Corr'] = safe_corr(gene_sig.value_histogram, mz_sig.value_histogram)
    scores['Spatial_Hist_Corr'] = safe_corr(gene_sig.spatial_histogram, mz_sig.spatial_histogram)
    scores['Radial_Corr'] = safe_corr(gene_sig.radial_profile, mz_sig.radial_profile)
    scores['Quadrant_Corr'] = safe_corr(gene_sig.quadrant_stats, mz_sig.quadrant_stats)
    scores['Morans_Sim'] = 1 / (1 + abs(gene_sig.morans_i - mz_sig.morans_i))
    
    return scores

# =============================================================================
# MAIN LOGIC
# =============================================================================

class RotationAnalyzer:
    def __init__(self, output_dir='./rotation_analysis_separated_v2'):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.rna = {}
        self.msi = {}

    def load_data(self):
        print("\n=== LOADING DATA ===")
        print(f"Checking RNA Folder: {RNA_INPUT_FOLDER}")
        if not os.path.exists(RNA_INPUT_FOLDER):
            print("!!! ERROR: RNA folder path does not exist!")
            
        loaded_rna = 0
        for f, s in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            p = os.path.join(RNA_INPUT_FOLDER, f)
            if os.path.exists(p): 
                self.rna[s] = sc.read_h5ad(p)
                loaded_rna += 1
            else:
                print(f"  Missing file: {f}")
        print(f"Successfully loaded {loaded_rna}/{len(RNA_SAMPLE_IDS)} RNA files.")

        print(f"\nChecking MSI Folder: {MSI_INPUT_FOLDER}")
        if not os.path.exists(MSI_INPUT_FOLDER):
            print("!!! ERROR: MSI folder path does not exist!")
            
        loaded_msi = 0
        for f, s in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            p = os.path.join(MSI_INPUT_FOLDER, f)
            if os.path.exists(p): 
                self.msi[s] = sc.read_h5ad(p)
                loaded_msi += 1
            else:
                print(f"  Missing file: {f}")
        print(f"Successfully loaded {loaded_msi}/{len(MSI_SAMPLE_IDS)} MSI files.")

    def get_signature(self, coords, vals, sid, name, ftype, k, aligned=None):
        nn = NearestNeighbors(n_neighbors=k+1).fit(coords)
        _, idx = nn.kneighbors(coords)
        loc_var = np.array([np.var(vals[idx[i,1:]]) for i in range(len(coords))])
        if loc_var.max()>loc_var.min(): loc_var=(loc_var-loc_var.min())/(loc_var.max()-loc_var.min())
        val_n = vals.copy()
        if val_n.max()>val_n.min(): val_n=(val_n-val_n.min())/(val_n.max()-val_n.min())
        imp = 0.5*loc_var + 0.5*val_n
        
        return SpatialSignature(
            sid, name, ftype, imp,
            compute_value_histogram(vals),
            compute_spatial_histogram(coords, vals),
            compute_radial_profile(coords, vals),
            compute_quadrant_stats(coords, vals),
            compute_morans_i(coords, vals, k),
            coords, vals, aligned
        )

    def run(self):
        print("\n=== STARTING ROTATION ANALYSIS ===")
        results = []
        
        if not self.rna:
            print("!!! STOPPING: No RNA data loaded.")
            return

        gene_avail = {}
        for g in AAD_TARGET_GENES:
            found_in_samples = {s: g in self.rna[s].var_names for s in RNA_SAMPLE_IDS if s in self.rna}
            gene_avail[g] = found_in_samples
            count = sum(found_in_samples.values())
            print(f"Gene '{g}' found in {count} samples.")
        
        # Check if we have any genes to process
        total_genes_found = sum(sum(v.values()) for v in gene_avail.values())
        if total_genes_found == 0:
            print("!!! STOPPING: None of the target genes were found in the loaded data.")
            return

        metric_keys = [
            'Value_Correlation', 'Importance_IoU', 'Importance_Correlation',
            'Value_Hist_Corr', 'Spatial_Hist_Corr', 'Radial_Corr',
            'Quadrant_Corr', 'Morans_Sim'
        ]

        # Loop angles
        for angle in range(0, 360, 5):
            print(f"Processing Angle {angle}°...")
            
            for gene in AAD_TARGET_GENES:
                samples = [s for s,a in gene_avail[gene].items() if a]
                
                for rs in samples:
                    if rs not in self.msi:
                        print(f"  Skipping {rs}: Found in RNA but missing in MSI data.")
                        continue
                    
                    # RNA Prep
                    r_adata = self.rna[rs]
                    r_coords = np.column_stack([r_adata.obs['x_um'], r_adata.obs['y_um']])
                    g_idx = list(r_adata.var_names).index(gene)
                    g_vals = r_adata.X[:, g_idx].toarray().flatten() if hasattr(r_adata.X, 'toarray') else r_adata.X[:, g_idx].flatten()
                    
                    # Rotate & Align
                    rot_coords = rotate_coords(r_coords, angle)
                    m_adata = self.msi[rs]
                    m_coords = np.column_stack([m_adata.obs['x_um'], m_adata.obs['y_um']])
                    aligned = align_bounds(rot_coords, m_coords)
                    
                    g_sig = self.get_signature(r_coords, g_vals, rs, gene, 'gene', 6, aligned)
                    
                    best_per_metric = {k: {'mz': None, 'score': -1.0} for k in metric_keys}
                    
                    m_vals_all = m_adata.X.toarray() if hasattr(m_adata.X, 'toarray') else m_adata.X
                    mz_names = list(m_adata.var_names)
                    
                    for i, mz_name in enumerate(mz_names):
                        mz_sig = self.get_signature(m_coords, m_vals_all[:, i], rs, mz_name, 'mz', 8)
                        scores = compute_all_scores(g_sig, mz_sig)
                        
                        for k in metric_keys:
                            if scores[k] > best_per_metric[k]['score']:
                                best_per_metric[k]['score'] = scores[k]
                                best_per_metric[k]['mz'] = mz_name
                    
                    row = {
                        'Rotation_Angle': angle,
                        'Gene': gene,
                        'Sample': rs
                    }
                    for k in metric_keys:
                        row[f'Best_MZ_{k}'] = best_per_metric[k]['mz']
                        row[f'Max_{k}'] = best_per_metric[k]['score']
                    
                    results.append(row)

        if results:
            df = pd.DataFrame(results)
            out_path = os.path.join(self.output_dir, 'rotation_scores_separate_v2.csv')
            df.to_csv(out_path, index=False)
            print(f"\nSUCCESS: Saved separated scores to: {out_path}")
            print(f"Rows generated: {len(df)}")
        else:
            print("\n!!! WARNING: Analysis finished but NO matches were found. CSV was not created.")


In [ ]:
if __name__ == "__main__":
    analyzer = RotationAnalyzer()
    analyzer.load_data()
    analyzer.run()


=== LOADING DATA ===
Checking RNA Folder: /home/ajarrah/PhD_Thesis/chapter_4/dummy_data_8/rna/
Successfully loaded 16/16 RNA files.

Checking MSI Folder: /home/ajarrah/PhD_Thesis/chapter_4/dummy_data_8/msi/
Successfully loaded 16/16 MSI files.

=== STARTING ROTATION ANALYSIS ===
Gene 'Gene_Gradient_X' found in 16 samples.
Gene 'Gene_Gradient_Y' found in 16 samples.
Gene 'Gene_Radial_Center' found in 16 samples.
Gene 'Gene_Blob_TopRight' found in 16 samples.
Gene 'Gene_Stripes_Vertical' found in 16 samples.
Gene 'Gene_Checkerboard' found in 16 samples.
Gene 'Gene_Ring_Donut' found in 16 samples.
Gene 'Gene_Spots_Grid' found in 16 samples.
Processing Angle 0°...
